### Semantic Chunking

Semantic Chunking is a document-splitting technique that groups related sentences together based on their meaning.

Instead of splitting text by a fixed number of characters or tokens, it uses embedding similarity to find where the topic or context changes.

This helps create chunks that are:

- Semantically meaningful
- Easy to understand
- Contextually connected
- Not cut off in the middle of an idea or thought

As a result, each chunk contains complete and coherent information, making it more effective for retrieval and processing tasks.

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

d:\RAG\RAG\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Initialise the model
model = SentenceTransformer('all-MiniLM-L6-v2')

## sample text
text = """
LangChain is a framework for building applications with LLMs.
LangChain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory and retrievers. The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

#Step1 : Split into Sentences
sentences = [s.strip() for s in text.split("\n") if s.strip()]

#Step2 : Embed each sentence

embeddings = model.encode(sentences)

#Step3 : Initialize parameters

threshold = 0.7 # control chunks tightness
chunks =[]
current_chunk = [sentences[0]]

#Step4 : Semantic grouping based on threshold 

for i in range(1, len(sentences)):
    sim = cosine_similarity(
        [embeddings[i-1]],
        [embeddings[i]]
    )[0][0]
    
    if sim >= threshold:
        current_chunk.append(sentences[i])
    else:
        chunks.append(" ".join(current_chunk))
        current_chunk = [sentences[i]]

# Append the last chunks

chunks.append(" ".join(current_chunk))

#Output the chunks

print("\n Semantic Chunks:")
for idx, chunks in enumerate(chunks):
    print(f"\nChunks {idx+1}:\n{chunks}")



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3714.23it/s]



 Semantic Chunks:

Chunks 1:
LangChain is a framework for building applications with LLMs. LangChain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.

Chunks 2:
You can create chains, agents, memory and retrievers. The Eiffel Tower is located in Paris.

Chunks 3:
France is a popular tourist destination.


### RAG Pipeline Modular code 

In [27]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableMap
from langchain_core.output_parsers import StrOutputParser


In [28]:
import os
from dotenv import load_dotenv

# Load .env first
load_dotenv()

# Check key
print(os.getenv("GROQ_API_KEY"))

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


In [29]:
### Custom Semantic Chunker with threshold 


class ThresholdSematicChunker:
    def __init__(self,model_name="all-MiniLM-L6-v2",threshold=0.7):
        self.model = SentenceTransformer(model_name)
        self.threshold = threshold
    
    def split(self, text:str):
        sentences = [s.strip() for s in text.split('.') if s.strip()]
        embeddings = self.model.encode(sentences)
        chunks=[]
        current_chunk = [sentences[0]]
        
        for i in range(1, len(sentences)):
            sim = cosine_similarity([embeddings[i -1]], [embeddings[i]]) [0][0]
            if sim >= self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(". ".join(current_chunk)+ ".")
                current_chunk = [sentences[i]]
        chunks.append(". ".join(current_chunk) + ".")
        return chunks
    
    def split_documents(self,docs):
        result = []
        for doc in docs:
            for chunk in self.split(doc.page_content):
                result.append(Document(page_content=chunk, metadata=doc.metadata))
        return result
                
        

In [30]:
## sample text
sample_text = """
LangChain is a framework for building applications with LLMs.
LangChain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory and retrievers. The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

doc = Document(page_content=sample_text)
doc

Document(metadata={}, page_content='\nLangChain is a framework for building applications with LLMs.\nLangChain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.\nYou can create chains, agents, memory and retrievers. The Eiffel Tower is located in Paris.\nFrance is a popular tourist destination.\n')

In [31]:
### Chunking 

chunker = ThresholdSematicChunker(threshold=0.7)
chunks = chunker.split_documents([doc])

chunks

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10296.57it/s]


[Document(metadata={}, page_content='LangChain is a framework for building applications with LLMs. LangChain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.'),
 Document(metadata={}, page_content='You can create chains, agents, memory and retrievers.'),
 Document(metadata={}, page_content='The Eiffel Tower is located in Paris.'),
 Document(metadata={}, page_content='France is a popular tourist destination.')]

In [32]:
### vector store
embedding = HuggingFaceEmbeddings()

vectorstore = FAISS.from_documents(chunks,embedding)
retriever = vectorstore.as_retriever()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5081.87it/s]


In [33]:
## Prompt  template

template = """Answer the question based on the following context:

{context}
Question: {question}
"""

prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based on the following context:\n\n{context}\nQuestion: {question}\n')

In [34]:
### LLM
llm = init_chat_model(
    "llama-3.1-8b-instant",
    model_provider="groq",
    temperature=0.4
)

### LCEL with retrieval 

rag_chain = (
    RunnableMap(
        {
            "context": lambda x: retriever.invoke(x["question"]),
            "question": lambda x: x["question"]
            
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

query = {"question":"What is Langchain used for?"}
result = rag_chain.invoke(query)

In [36]:
print(result)



LangChain is a framework for building applications with LLMs (Large Language Models). It provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.


### Semantic Chunker with Langchain

In [41]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.document_loaders import TextLoader

In [42]:
## Load the documents
loader = TextLoader("langchain_into.txt")
docs =loader.load()

## Initialize embedding model 
embedding = HuggingFaceEmbeddings(
      model_name="sentence-transformers/all-MiniLM-L6-v2"
)

## create the semantic Chunker
chunker = SemanticChunker(embedding)

## Split the document
chunks = chunker.split_documents(docs)

## Result 
for i, chunk in enumerate(chunks):
    print(f"\n Chunk {i+1}:\n{chunk.page_content} ")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9362.69it/s]



 Chunk 1:
LangChain is a framework for building applications powered by Large Language Models (LLMs). It provides modular components for prompts, chains, agents, memory, and retrieval. Developers can integrate models from OpenAI, Anthropic, and other providers. Retrieval-Augmented Generation (RAG) is a common use case in LangChain. Vector databases such as Pinecone and Chroma are often used to store embeddings. Machine learning models learn patterns from data. Supervised learning requires labeled examples. Unsupervised learning discovers hidden structures in datasets. Deep learning uses neural networks with many layers. Training large models can require significant computational resources. The Eiffel Tower is located in Paris. Millions of tourists visit Paris every year. France is known for its history, art, and cuisine. The Louvre Museum houses famous artworks including the Mona Lisa. French is the official language of France. My coffee mug is blue and sits on the corner of the desk.

# Semantic Chunker Working Flow

```text
docs
 │
 ▼
SemanticChunker
 │
 ▼
Text ko sentences me split karta hai
 │
 ├── Sentence 1
 ├── Sentence 2
 ├── Sentence 3
 │
 ▼
Embedding Model
 │
 ▼
Sentence → Vector
 │
 ▼
Cosine Similarity
 │
 ▼
Similar Sentences Group
 │
 ▼
Final Chunks
```

## Step-by-Step

### 1. Load Document

```python
docs = loader.load()
```

Document text load hota hai.

---

### 2. Sentence Splitting

SemanticChunker text ko sentences me split karta hai.

```text
Sentence 1
Sentence 2
Sentence 3
```

---

### 3. Generate Embeddings

Embedding model har sentence ko vector me convert karta hai.

```text
Sentence 1 → [0.12, 0.45, ...]
Sentence 2 → [0.11, 0.47, ...]
Sentence 3 → [0.78, -0.21, ...]
```

---

### 4. Calculate Similarity

Cosine Similarity calculate hoti hai.

```text
S1 ↔ S2 = 0.95
S2 ↔ S3 = 0.20
```

---

### 5. Create Chunks

High similarity wale sentences ek group me aa jate hain.

```text
Chunk 1:
Sentence 1
Sentence 2

Chunk 2:
Sentence 3
```

## Summary

SemanticChunker:
1. Text ko sentences me split karta hai.
2. Embeddings generate karta hai.
3. Cosine similarity calculate karta hai.
4. Similar meaning wale sentences ko ek chunk me group karta hai.
```